In [ ]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, Model
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
import matplotlib.pyplot as plt

# --- 1. Configuration ---
SYMBOL = 'ETHUSDT'
MODEL_TYPE = 'transformer'
EPOCHS = 50
BATCH_SIZE = 64

# --- 2. Local VS Code Pathing ---
BASE_DIR = os.path.abspath(os.getcwd())
DATA_DIR = os.path.join(BASE_DIR, f'processed_data_{MODEL_TYPE}', SYMBOL)
MODEL_SAVE_PATH = os.path.join(BASE_DIR, f'{MODEL_TYPE}_model_{SYMBOL}.keras')

print(f"📁 Loading {SYMBOL} data from:\n{DATA_DIR}")

# --- 3. Load Data ---
try:
    X_train = np.load(os.path.join(DATA_DIR, 'X_train.npy'))
    y_train = np.load(os.path.join(DATA_DIR, 'y_train.npy'))
    X_val = np.load(os.path.join(DATA_DIR, 'X_val.npy'))
    y_val = np.load(os.path.join(DATA_DIR, 'y_val.npy'))
    X_test = np.load(os.path.join(DATA_DIR, 'X_test.npy'))
    y_test = np.load(os.path.join(DATA_DIR, 'y_test.npy'))
    print("✅ Data successfully loaded!")
    print(f"X_train shape: {X_train.shape}")
except FileNotFoundError as e:
    print(f"❌ Error loading data: {e}\nPlease run the preprocessing script for {SYMBOL} first.")

# --- 4. Build Transformer Architecture ---
def transformer_encoder(inputs, head_size, num_heads, ff_dim, dropout=0):
    # Attention and Normalization
    x = layers.MultiHeadAttention(key_dim=head_size, num_heads=num_heads, dropout=dropout)(inputs, inputs)
    x = layers.Dropout(dropout)(x)
    x = layers.LayerNormalization(epsilon=1e-6)(x)
    res = x + inputs

    # Feed Forward Part
    x = layers.Conv1D(filters=ff_dim, kernel_size=1, activation="relu")(res)
    x = layers.Dropout(dropout)(x)
    x = layers.Conv1D(filters=inputs.shape[-1], kernel_size=1)(x)
    x = layers.LayerNormalization(epsilon=1e-6)(x)
    return x + res

def build_transformer(input_shape):
    inputs = layers.Input(shape=input_shape)

    # Transformer Block
    x = transformer_encoder(inputs, head_size=64, num_heads=4, ff_dim=64, dropout=0.1)

    # Global pooling to flatten the sequence
    x = layers.GlobalAveragePooling1D()(x)

    # MLP Head
    x = layers.Dense(32, activation="relu")(x)
    x = layers.Dropout(0.1)(x)
    outputs = layers.Dense(1, activation="linear")(x)

    model = Model(inputs, outputs)
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4), loss="mse", metrics=["mae"])
    return model

model = build_transformer((X_train.shape[1], X_train.shape[2]))
model.summary()

# --- 5. Callbacks & Training ---
callbacks = [
    EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True),
    ModelCheckpoint(MODEL_SAVE_PATH, monitor='val_loss', save_best_only=True)
]

print(f"\n🚀 Starting training for {SYMBOL}...")
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks,
    verbose=1
)

# --- 6. Plot Training History ---
plt.figure(figsize=(10, 4))
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title(f'{SYMBOL} Transformer Training History')
plt.xlabel('Epochs')
plt.ylabel('Loss (MSE)')
plt.legend()
plt.show()

# --- 7. Evaluation ---
test_loss, test_mae = model.evaluate(X_test, y_test, verbose=0)
print(f"🎯 Test MAE: {test_mae:.5f}")
print(f"💾 Best model saved to: {MODEL_SAVE_PATH}")